<div style='background:#0b0e10;color:#f4f5f6;border:1px solid #2c3136;border-radius:18px;padding:30px 24px;text-align:center'><div style='font-family:monospace;font-size:12px;letter-spacing:.22em;color:#9aa1aa'>OFFICIAL COLAB · v1.7.0</div><h1 style='font-size:38px;margin:12px 0 4px'>MINEIRO USERNAME INTELLIGENCE</h1><p style='font-size:18px;color:#a3a9b0;margin:0'>OSINT Investigation Workbench</p><p style='font-size:13px;color:#7f8790;margin-top:14px'>Collection → Evidence → Correlation → Assessment</p></div>

**Uso:** Runtime → Run all. Ao final, abra o link exibido em **Mineiro Username Intelligence**.

> O notebook executa a aplicação na VM temporária do Colab (build de produção, sem servidor de desenvolvimento e sem integrações de IA). Ele não é hospedagem permanente.

## 1 · Environment
Configuração canônica. Normalmente você não precisa alterar nada.

In [ ]:
from pathlib import Path
from IPython.display import display, HTML

REPO_URL = 'https://github.com/Ridd1kulusC0d3r/mineirohml.git'
BRANCH = 'main'  # antes do merge do PR de refatoração, use 'refactor/mineiro-tool'
SUBDIR = 'mineiro-tool'
CLONE = Path('/content/mineirohml')
ROOT = CLONE / SUBDIR
PORT = 3000
LOG_FILE = Path('/content/mineiro-server.log')
NOTEBOOK_VERSION = '1.7.0'
COMPAT = False  # troque para True se o link do Colab der HTTP 500 (desliga cabeçalhos estritos e cache condicional)

display(HTML(f"""
<div style='border:1px solid #ddd;border-radius:12px;padding:14px 16px'>
<b>Mineiro Username Intelligence Official Colab</b><br>
<span style='color:#666'>branch: {BRANCH} · port: {PORT} · notebook: {NOTEBOOK_VERSION}</span>
</div>"""))

## 2 · Bootstrap
Clona só a pasta da ferramenta (checkout parcial), garante Node.js 22+, instala dependências com `npm ci`, valida o Registry e gera o build de produção.

In [ ]:
import os, subprocess, re

def run(cmd, cwd=None):
    print('$', ' '.join(cmd))
    subprocess.run(cmd, cwd=cwd, check=True)

if CLONE.exists():
    run(['git', 'fetch', '--depth', '1', 'origin', BRANCH], cwd=CLONE)
    run(['git', 'checkout', '-B', BRANCH, 'FETCH_HEAD'], cwd=CLONE)
else:
    run(['git', 'clone', '--depth', '1', '--filter=blob:none', '--sparse', '--branch', BRANCH, REPO_URL, str(CLONE)])
    run(['git', 'sparse-checkout', 'set', SUBDIR], cwd=CLONE)

node_version = subprocess.check_output(['node', '--version'], text=True).strip()
major = int(re.sub(r'[^0-9].*', '', node_version.lstrip('v')))
if major < 22:
    print(f'Node {node_version} detectado. Atualizando para Node 22...')
    run(['npm', 'install', '-g', 'n'])
    run(['n', '22'])

run(['npm', 'ci', '--no-audit', '--no-fund', '--prefer-offline'], cwd=ROOT)
run(['npm', 'run', 'registry:validate'], cwd=ROOT)
run(['npm', 'run', 'build'], cwd=ROOT)

print('\n[OK] Ambiente preparado em', ROOT)

## 3 · Start Mineiro
Inicia o servidor de produção (`node dist/server.cjs`, escutando em `0.0.0.0` dentro da VM isolada do Colab), espera o health check e abre a UI pelo proxy oficial do Colab.

In [ ]:
import os, re, subprocess, time, requests
from google.colab import output
from IPython.display import display, HTML, Javascript

try:
    if 'mineiro_server' in globals() and mineiro_server.poll() is None:
        mineiro_server.terminate()
        mineiro_server.wait(timeout=10)
except Exception:
    pass

# Libera a porta se um servidor antigo (de uma execução anterior) ainda a ocupa.
try:
    import psutil
    for conn in psutil.net_connections(kind='tcp'):
        if conn.status == 'LISTEN' and conn.laddr.port == PORT and conn.pid:
            print(f'Porta {PORT} ocupada pelo PID {conn.pid}; encerrando.')
            psutil.Process(conn.pid).terminate()
except Exception as exc:
    print('Aviso: não foi possível verificar a porta:', exc)
time.sleep(1)

log_handle = open(LOG_FILE, 'w', buffering=1)
mineiro_server = subprocess.Popen(
    ['node', 'dist/server.cjs'],
    cwd=ROOT,
    # O proxy do Colab não alcança o loopback (HTTP 500). A VM é isolada e temporária, então 0.0.0.0 é seguro aqui.
    env={**os.environ, 'NODE_ENV': 'production', 'PORT': str(PORT), 'HOST': '0.0.0.0', 'MINEIRO_COMPAT_HEADERS': '1' if COMPAT else '0'},
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    text=True,
    start_new_session=True,  # desanexa do grupo de processos da célula
)

health = None
for _ in range(30):
    try:
        health = requests.get(f'http://127.0.0.1:{PORT}/api/health', timeout=2)
        if health.ok:
            break
    except Exception:
        pass
    time.sleep(1)

# O proxy do Colab chega pela interface de rede da VM, não pelo loopback: confira por lá também.
import socket
vm_ip = socket.gethostbyname(socket.gethostname())
page = requests.get(f'http://{vm_ip}:{PORT}/', timeout=5)
asset = re.search(r'/assets/[^"\']+\.js', page.text)
asset_ok = (not asset) or requests.get(f'http://{vm_ip}:{PORT}{asset.group(0)}', timeout=5).ok
print(f'servidor vivo: {mineiro_server.poll() is None} · página: {page.status_code} · asset: {asset_ok} · ip: {vm_ip}')
if mineiro_server.poll() is not None or not page.ok or not asset_ok:
    print(LOG_FILE.read_text()[-6000:])
    raise RuntimeError('Mineiro não responde pela interface de rede da VM. Veja o log acima.')

if not health or not health.ok:
    print(LOG_FILE.read_text()[-6000:] if LOG_FILE.exists() else 'Sem log disponível.')
    raise RuntimeError('Mineiro não ficou saudável dentro do tempo esperado.')

proxy_url = output.eval_js(f'google.colab.kernel.proxyPort({PORT})')
payload = health.json()
display(HTML(f"""
<div style='border:1px solid #cfd3d7;border-radius:14px;padding:18px'>
<div style='font-size:12px;color:#666;letter-spacing:.12em'>ENVIRONMENT READY</div>
<div style='font-size:20px;font-weight:700;margin:6px 0'>Mineiro Username Intelligence</div>
<a href='{proxy_url}' target='_blank' style='display:inline-block;background:#111;color:#fff;padding:10px 16px;border-radius:8px;text-decoration:none'>Abrir Mineiro</a>
<div style='margin-top:10px;color:#777;font-size:12px'>health: {payload}</div>
</div>"""))
display(Javascript(f"window.open('{proxy_url}', '_blank');"))

## 4 · Validation opcional
Use para validar a demo sintética e a camada analítica sem consultar pessoas reais.

In [ ]:
run(['npm', 'run', 'demo'], cwd=ROOT)
run(['npm', 'run', 'intelligence:test'], cwd=ROOT)

## 5 · Diagnostics
Rode esta célula **logo depois** de ver tela escura ou HTTP 500. Ela mostra se o servidor está vivo, se responde pela rede da VM e as últimas linhas do log.


In [ ]:
import os, shutil, socket, subprocess
import requests

print('== processo ==')
alive = 'mineiro_server' in globals() and mineiro_server.poll() is None
print('servidor vivo:', alive, '' if alive or 'mineiro_server' not in globals() else f'(código de saída {mineiro_server.poll()})')
print('node:', subprocess.run(['node', '--version'], capture_output=True, text=True).stdout.strip())
print(subprocess.run(['bash', '-c', 'free -m | head -2; echo; (ss -ltnp 2>/dev/null || netstat -ltnp 2>/dev/null) | grep -E ":%d\\b" || echo "nada escutando na porta %d"' % (PORT, PORT)], capture_output=True, text=True).stdout)

print('== testes a partir da VM ==')
for label, host in (('loopback', '127.0.0.1'), ('rede da VM', socket.gethostbyname(socket.gethostname()))):
    try:
        r = requests.get(f'http://{host}:{PORT}/api/health', timeout=4)
        print(f'{label} ({host}): HTTP {r.status_code}')
    except Exception as exc:
        print(f'{label} ({host}): FALHOU -> {exc}')

print('\n== últimas 100 linhas do log (inclui erros do navegador: [client:...]) ==')
if LOG_FILE.exists():
    print('\n'.join(LOG_FILE.read_text(errors='replace').splitlines()[-100:]))
else:
    print('Log ainda não existe.')


## 6 · Stop
Encerra o servidor desta sessão.

In [ ]:
if 'mineiro_server' in globals() and mineiro_server.poll() is None:
    mineiro_server.terminate()
    mineiro_server.wait(timeout=10)
    print('Mineiro encerrado.')
else:
    print('Nenhum servidor ativo nesta sessão.')